#Import

- import libraries



- Импорт Библиотек для работы


In [ ]:
import numpy as np
import pandas as pd

# Remarks
- # RU → EN:
- Склад — warehouse
- Количество продано — sales quantity
- Общий остаток — total stock
- Проданный товар — sold items

#File load

- your file load
- загрузка файла для работы

In [ ]:
df = pd.read_excel('write adress') #введите свой путь для загрузки файла


#Data prepare

- prepare data
- подготовка данных

In [ ]:
# Prepare and aggregate required columns
# Подготовка нужных колонок
def prepare_df(df):
    df = df.copy().fillna(0)
    df['товар в приходе'] = df[['Доступно 7 дней','Доступно 14 дней','Доступно 21 день','Доступно 28 дней']].sum(axis=1)
    df['Товар на складе'] = df[['В наличии','В резерве','Доступно','Приход','товар в приходе','Остаток']].sum(axis=1)
    df['Проданный товар'] = df['Отгружается'] + df['Расход']
    df['Общий остаток'] = df['Товар на складе'] - df['Проданный товар']
    return df[['Склад','Количество продано','Общий остаток','Проданный товар']]

# Metrics calculation

- calculation metric
- просчет метрик

In [ ]:
# Calculate metrics: average sales, order level, and demand spikes
# Просчет метрик под задачу
def calc_metrics(df, anomaly_threshold=2, season_months=3, season_multi=2):
    df = df.copy()
    df['Среднее'] = df['Количество продано'] / season_months
    df['Среднее для заказа'] = (df['Среднее'] * season_multi).round(0)
    df['Всплеск продаж'] = np.where(df['Проданный товар'] > (df['Среднее'] * anomaly_threshold), 1, 0)
    return df

# Order calculation

- Order calculation
- Подсчет заказа

In [ ]:
# Calculate required order volume based on stock and demand
# Просчет заказа на основе нужных данных
def order(df):
    df = df.copy()
    df['Заказ'] = np.where(
        df['Общий остаток'] < df['Среднее для заказа'],
        df['Среднее для заказа'] - df['Общий остаток'],
        0
    )
    return df[['Склад','Заказ','Всплеск продаж']]

# Result and test
- result script
- финальный результат и запуск

In [ ]:
result = order(calc_metrics(prepare_df(df), season_months=3, season_multi=2))
result.head(15)